# Simulação de Rede de Agulhas de Bússola (Dipolos Magnéticos 2D)

Este Jupyter Notebook adapta o script [compass_sim_gpu.py](file:///Users/jps/Dropbox/simula%20agulhas/compass_sim_gpu.py) para que possa ser executado e explorado de forma interativa. Aqui você poderá rodar a simulação física, alterar parâmetros geométricos e dinâmicos, e visualizar os gráficos e vídeos resultantes diretamente na interface do Notebook.

A simulação modela uma grade 2D de dipolos magnéticos clássicos (agulhas de bússola) interagindo pelo campo magnético dipolar clássico. A dinâmica é integrada usando o algoritmo *Velocity-Verlet* inercial com amortecimento viscoso do ar.

## 1. Verificação de Ambiente e Dependências

Primeiro, vamos verificar a disponibilidade das dependências fundamentais e a presença de aceleração de hardware (GPU NVIDIA via CUDA/CuPy) e do `ffmpeg` para geração de vídeo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shutil
import os

# Tenta importar o módulo de simulação
try:
    import compass_sim_gpu as csg
    print("Módulo 'compass_sim_gpu' importado com sucesso!")
    print(f"  - GPU Disponível para cálculos: {csg._GPU_AVAILABLE}")
    if not csg._GPU_AVAILABLE:
        print(f"  - Detalhe/Erro do backend de GPU: {csg._GPU_ERROR_MSG}")
except ImportError as e:
    print(f"Erro ao importar compass_sim_gpu: {e}")

# Verifica ffmpeg
ffmpeg_path = shutil.which('ffmpeg')
if ffmpeg_path:
    print(f"ffmpeg encontrado em: {ffmpeg_path} (Geração de vídeos habilitada)")
else:
    print("AVISO: ffmpeg não encontrado no PATH. A geração de vídeos MP4 não funcionará.")
    print("  Instale via: brew install ffmpeg (macOS) ou apt install ffmpeg (Linux)")

## Otimização Recente: Pré-computação das Matrizes de Acoplamento

Para aumentar drasticamente o desempenho de grandes simulações, o código calcula as interações dipolares de forma otimizada. 

Como as posições físicas das agulhas da grade ($x_i, y_i$) são fixas ao longo de toda a execução, a geometria de distâncias e os termos de acoplamento da interação magnética também são constantes. Em vez de recalcular tudo a cada passo da integração (Complexidade $O(K^2)$ com loops aninhados para as réplicas periódicas e operações de raiz quadrada), o código pré-calcula as matrizes de acoplamento estático ($C_{xx}, C_{xy}, C_{yy}$) uma única vez no início do relaxamento.

Com isso, cada passo do loop quente de simulação realiza apenas **multiplicações de matriz-vetor** ultra-rápidas usando sub-rotinas de álgebra linear altamente otimizadas (BLAS na CPU e cuBLAS na GPU):
$$\vec{B}_{dip, x} = C_{xx} \cdot \vec{m}_x + C_{xy} \cdot \vec{m}_y$$
$$\vec{B}_{dip, y} = C_{xy} \cdot \vec{m}_x + C_{yy} \cdot \vec{m}_y$$

Isso elimina a sobrecarga de alocação de memória no loop quente e acelera o relaxamento, permitindo simular grades densas de agulhas de forma instantânea.

## 2. Método 1: Execução do Script via Comando `%run` (Estilo CLI)

A forma mais direta de rodar o programa é executá-lo diretamente como se estivesse no terminal, usando o comando mágico `%run`. Isso executa a função `main()` do script com os argumentos especificados.

Ao rodar com `%run`, todas as variáveis globais criadas pelo script (como `xs`, `ys`, `thetas_init`, `thetas_eq`, etc.) serão carregadas no escopo interativo do notebook para análise posterior.

In [ ]:
# Executa a simulação para uma grade de 6x6 na geometria triangular por 1.5 segundos físicos.
# Será gerado um vídeo chamado 'simulacao_notebook.mp4' (se o ffmpeg estiver disponível)
# e os arquivos de imagens correspondentes.

%run compass_sim_gpu.py --geometry triangular --N 6 --M 6 --t_sim 1.5 --damping 1e-7 --video simulacao_notebook.mp4

### Exibição dos Resultados Gerados via `%run`

Abaixo, carregamos as imagens de saída e o vídeo gerado para exibição direta dentro deste Notebook.

In [ ]:
from IPython.display import Image, display, Video

print("=== Estado Inicial vs Estado de Equilíbrio ===")
if os.path.exists("compass_comparison.png"):
    display(Image(filename="compass_comparison.png"))
else:
    print("Arquivo 'compass_comparison.png' não encontrado.")

print("=== Parâmetro de Ordem Global S(t) ===")
if os.path.exists("compass_order_param.png"):
    display(Image(filename="compass_order_param.png"))
else:
    print("Arquivo 'compass_order_param.png' não encontrado.")

### Exibição do Vídeo MP4 Inline

Se o vídeo foi gerado com sucesso, execute a célula abaixo para assisti-lo embutido no Notebook.

In [ ]:
# O script adiciona um sufixo numérico de versão automática ao vídeo (ex: simulacao_notebook0000.mp4)
# Vamos procurar o vídeo mais recente gerado que combine com o prefixo 'simulacao_notebook'
import glob
videos = sorted(glob.glob("simulacao_notebook*.mp4"))

if videos:
    latest_video = videos[-1]
    print(f"Exibindo vídeo: {latest_video}")
    display(Video(latest_video, embed=True, width=640))
else:
    print("Nenhum vídeo 'simulacao_notebook*.mp4' encontrado. Certifique-se de que o ffmpeg está instalado.")

## 3. Método 2: Execução Programática (APIs Python)

Em vez de rodar o script como um comando de terminal, podemos importar suas funções individuais e chamá-las diretamente. Isso é ideal para construir fluxos de trabalho personalizados, análises de dados ou loops de varredura.

Vamos realizar uma simulação programática completa passo a passo:

In [ ]:
import compass_sim_gpu as csg
import matplotlib.pyplot as plt

# 1. Definição de Parâmetros Geométricos
N, M = 8, 8
geometry = 'honeycomb'  # square, triangular, honeycomb
R = 0.025               # raio visual [m]
noise = 1.5             # amplitude do ruído inicial [rad]
seed = 42

np.random.seed(seed)

# 2. Geração da Grade de Agulhas
xs, ys, thetas_init, nn_dist, Lx_period, Ly_period = csg.make_grid(
    N=N, M=M, geometry=geometry, noise=noise, R=R
)

# 3. Plot do Estado Inicial
# plot_state retorna (fig, ax) que é exibido diretamente pelo Jupyter
needle_len = 0.80 * 2.0 * R
needle_width = needle_len * 0.22

fig_init, ax_init = csg.plot_state(
    thetas_init, xs, ys, 
    title="Estado Inicial (Geometria Honeycomb)",
    needle_len=needle_len, needle_width=needle_width
)
plt.show()  # força renderização explícita

### Execução da Integração Dinâmica

Agora chamamos a função `relax` para rodar a simulação física com acoplamento dipolar total. Vamos rodar em GPU caso disponível.

In [ ]:
# Amortecimento um pouco maior para um equilíbrio rápido e estável
damping = 8e-8 
t_sim = 2.0
cutoff = nn_dist * 2.6  # corte físico para agrupar vizinhos próximos
moment = csg.compute_moment_from_geometry(needle_len, needle_width, csg.NEEDLE_THICKNESS_DEFAULT)
inertia = csg.compute_inertia_from_geometry(needle_len, needle_width, csg.NEEDLE_THICKNESS_DEFAULT)

print("Iniciando relaxamento dinâmico programático...")
thetas_eq, omegas_eq, thetas_hist, n_frames, sim_dt, stop_reason, field_log = csg.relax(
    thetas_init.copy(), xs, ys,
    t_sim=t_sim,
    inertia=inertia,
    damping=damping,
    cutoff=cutoff,
    moment=moment,
    use_gpu=csg._GPU_AVAILABLE,
    show_progress=True,  # exibe progresso no notebook
    make_images=False    # não salva PNGs intermediários em disco durante o cálculo
)

print(f"\nSimulação concluída! Motivo do término: {stop_reason}")

### Plotagem do Estado Final de Equilíbrio

Visualizamos a rede resultante após o decaimento dinâmico.

In [ ]:
fig_eq, ax_eq = csg.plot_state(
    thetas_eq, xs, ys, 
    title=f"Estado de Equilíbrio - Honeycomb ({N}x{M})",
    needle_len=needle_len, needle_width=needle_width
)
plt.show()

### Evolução do Parâmetro de Ordem Global

Extraímos o histórico angular e plotamos $S(t)$ usando a função de plot do próprio módulo.

In [ ]:
# O plot_order_parameter salva um arquivo em disco, mas como ele retorna a figura
# podemos exibi-la inline de imediato.
csg.plot_order_parameter(
    [th for th, _ in thetas_hist],
    outpath="notebook_order_param.png",
    dt=sim_dt
)
display(Image("notebook_order_param.png"))

## 4. Estudo de Campo Magnético Externo e Ciclo de Histerese

O script suporta o modo `'hysteresis'` onde o campo externo oscila lentamente ao longo do tempo. Esse modo é ideal para investigar a resposta coletiva da rede de bússolas, simulando o clássico fenômeno de histerese ferromagnética.

In [ ]:
# Roda uma varredura de histerese rápida com o comando mágico %run
# --B_ext de 10 mT e tempo total de 4.0s para cobrir o ciclo completo

%run compass_sim_gpu.py --geometry square --N 6 --M 6 --field_mode hysteresis --B_ext 10e-3 --t_sim 4.0

### Curva de Histerese Magnética resultante

Abaixo exibimos a curva gerada de magnetização versus campo aplicado ($M \times B$).

In [ ]:
if os.path.exists("hysteresis_loop.png"):
    display(Image(filename="hysteresis_loop.png"))
else:
    print("Arquivo 'hysteresis_loop.png' não encontrado. Verifique se o modo de campo 'hysteresis' foi rodado.")